# Feature engineering

Tweet text is vectorized into a TF-IDF matrix, capping vocabulary
size explicitly (`max_features`) and sticking to unigrams and
bigrams, which keeps the resulting matrix a compact, usable size for
classical ML models trained on a 20,000-tweet sample.

In [1]:
import os
import re
import pandas as pd
import numpy as np

train = pd.read_pickle('data/train_sample.pkl')
test_binary = pd.read_pickle('data/test_binary.pkl')

In [2]:
def clean_tweet(text):
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text

train['clean_text'] = train.text.apply(clean_tweet)
test_binary['clean_text'] = test_binary.text.apply(clean_tweet)
train[['text', 'clean_text']].head()

,text,clean_text
0,@DawnRichard THANKS LUV...KEEP THE GRIND GOING...,thanks luv keep the grind going like i know u ...
1,I feel like watching The Life Aquatic. But I d...,i feel like watching the life aquatic but i do...
2,So aparentally early this morning Erol died of...,so aparentally early this morning erol died of...
3,"Chopchop, get dressed! Getting ready for work",chopchop get dressed getting ready for work
4,@misshaleymae You know it baby ! x,you know it baby x


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features = 5000, ngram_range = (1, 2), min_df = 2)
X_train = vectorizer.fit_transform(train.clean_text)
X_test = vectorizer.transform(test_binary.clean_text)

print(f'vocabulary size: {len(vectorizer.vocabulary_)}')
X_train.shape, X_test.shape

vocabulary size: 5000


((20000, 5000), (359, 5000))

In [4]:
from sklearn.feature_selection import chi2

y_train = train.label.values
chi2_scores, _ = chi2(X_train, y_train)
feature_names = np.array(vectorizer.get_feature_names_out())
top_idx = np.argsort(chi2_scores)[::-1][:15]
pd.Series(chi2_scores[top_idx], index = feature_names[top_idx])

thanks       93.986887
sad          72.204863
miss         59.510345
love         49.949587
you          48.997806
not          41.820313
thank        38.436880
wish         37.810834
sick         37.598485
thank you    36.340049
sucks        35.200017
bad          34.965466
work         34.559861
no           32.500676
hate         32.393896
dtype: float64

In [5]:
import pickle
os.makedirs('data', exist_ok = True)
with open('data/model_matrix.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train, 'y_train': y_train,
        'X_test': X_test, 'y_test': test_binary.label.values,
        'vectorizer_vocab_size': len(vectorizer.vocabulary_),
    }, f)

with open('data/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

import json
with open('outputs/feature_engineering_summary.json', 'w') as f:
    json.dump({
        'vocab_size': len(vectorizer.vocabulary_),
        'top_chi2_word': feature_names[top_idx[0]],
    }, f, indent = 2)
X_train.shape

(20000, 5000)